# LogisChain AI — 04: Financial Integration

Demonstrate how supply chain intelligence improves financial risk models.

**Goals:**
- Price trade finance instruments with and without SC adjustment
- Predict CCC from SC signals
- Run the full credit risk scorer
- Quantify SC contribution via SHAP decomposition

In [2]:
!rm -rf /content/logischain-ai-8
!git clone https://github.com/ZethetaIntern/logischain-ai-8.git -q
!pip install mlflow lightgbm shap optuna tqdm faker lifelines -q
import sys
sys.path.insert(0, '/content/logischain-ai-8')
print('Setup done!')

Setup done!


In [3]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data.pipeline import SyntheticDataGenerator
from src.features.fusion_features import FeaturePipeline
from src.financial.trade_finance_model import TradeFinanceRiskModel, TradeFinanceInstrument
from src.financial.ccc_predictor import CCCPredictor
from src.financial.credit_risk_scorer import SupplyChainCreditScorer

gen = SyntheticDataGenerator(seed=42)
data = gen.generate_all()
fp = FeaturePipeline()
fused = fp.run(data['carriers'], data['shipments'], data['financial'])
print(f'Fused features: {fused.shape}')

Fused features: (500, 86)


In [4]:
# Trade Finance Pricing
tf_model = TradeFinanceRiskModel()
instruments = [
    TradeFinanceInstrument('LC-001', 'LC', 1_000_000, 90, 0.05, 'A', 'BBB', '84', disruption_probability=0.05),
    TradeFinanceInstrument('LC-002', 'LC', 1_000_000, 90, 0.05, 'A', 'BBB', '84', disruption_probability=0.40),
    TradeFinanceInstrument('SCF-001', 'SCF', 5_000_000, 60, 0.06, 'BB', 'B', '85', disruption_probability=0.25),
]
portfolio = tf_model.price_portfolio(instruments)
print(portfolio[['instrument_id', 'spread_bps', 'pd_estimate', 'expected_loss_usd', 'sc_disruption_premium_bps']])

  instrument_id  spread_bps  pd_estimate  expected_loss_usd  \
0        LC-001        85.0       0.0055             2475.0   
1        LC-002       260.0       0.0230            10350.0   
2       SCF-001       369.0       0.0475           106875.0   

   sc_disruption_premium_bps  
0                       25.0  
1                      200.0  
2                      125.0  


In [5]:
# CCC Prediction
if 'cash_conversion_cycle' in fused.columns:
    ccc_predictor = CCCPredictor(model_type='gbm')
    ccc_predictor.fit(fused.dropna(subset=['cash_conversion_cycle']))
    eval_metrics = ccc_predictor.evaluate(fused.dropna(subset=['cash_conversion_cycle']))
    print('CCC Predictor metrics:', eval_metrics)
    # Shock simulation
    baseline, shocked, delta = ccc_predictor.simulate_sc_shock(fused.dropna(subset=['cash_conversion_cycle']), delay_increase_days=15)
    print(f'Average CCC increase from 15-day delay shock: {np.mean(delta):.2f} days')

In [6]:
# Credit Risk Scoring
if 'default_flag' in fused.columns:
    scorer = SupplyChainCreditScorer()
    scorer.fit(fused.dropna(subset=['default_flag']))
    eval_metrics = scorer.evaluate(fused.dropna(subset=['default_flag']))
    print('Credit Scorer metrics:', eval_metrics)

    results = scorer.score_entities(fused.head(50))
    summary = scorer.portfolio_expected_loss(results)
    print('\nPortfolio EL Summary:', summary)
    print(f'Average SC contribution to PD: {summary["avg_sc_contribution"]:.1%}')

Credit Scorer metrics: {'roc_auc': 1.0, 'brier_score': 0.01559178895760895, 'mean_pd': 0.08263870636525689}

Portfolio EL Summary: {'total_ead_usd': 50000000.0, 'total_expected_loss_usd': 2183441.45, 'el_ratio': 0.043669, 'avg_pd': 0.097042, 'tier_distribution': {'LOW': 45, 'MEDIUM': 3, 'HIGH': 1, 'CRITICAL': 1}, 'avg_sc_contribution': 0.0034}
Average SC contribution to PD: 0.3%
